# NHL model evaluation

This notebook inspects artifacts from the canonical `src/train_model.py` pipeline.
Rebuild features and run that script first using the documented Python environment.
The training script uses NHL win flags (including shootouts), selects features and
regularization on earlier seasons, and evaluates the latest season separately.
The production model is subsequently refitted on all eligible games; its in-sample
predictions must not be reported as holdout performance.


In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "NHL_Game_Data"]
project = next(p for p in candidates if (p / "src/train_model.py").exists())
report = json.loads((project / "models/evaluation.json").read_text())
report["selected"]


In [ ]:
pd.DataFrame({"model": report["holdout"]["model"],
              "training-prior baseline": report["holdout"]["baseline"]})


In [ ]:
pd.DataFrame([{ "feature_set": c["feature_set"], "C": c["C"],
                "validation_log_loss": c["mean_validation_log_loss"] }
              for c in report["candidates"]]).sort_values("validation_log_loss")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(plt.imread(project / "models/calibration.png"))
ax.axis("off")
plt.show()


In [ ]:
# Standardized coefficients from the final model, not the evaluation model.
pd.read_csv(project / "models/feature_coefficients.csv")
